# CrowdSense ML - Notebook 02: Continuous Density Map Generation

This notebook demonstrates how discrete head-point annotations $\mathbf{x}_i$ are converted into continuous Gaussian density distributions $D(x)$ such that:

$$\sum_{x, y} D(x, y) = N$$

### Formulation:
$$D(p) = \sum_{i=1}^{N} \delta(p - x_i) * G_{\sigma_i}(p)$$

For congested scenes (**Part A**), we utilize **Geometry-Adaptive Gaussian Kernels**:
$$\sigma_i = \beta \bar{d}_i$$
where $\bar{d}_i$ is the average distance of head $i$ to its $k=3$ nearest neighbors, and $\beta = 0.3$.

In [ ]:
import numpy as np
from scipy.spatial import KDTree
import matplotlib.pyplot as plt

def generate_density_map(points, shape, beta=0.3, k=3):
    h, w = shape
    density = np.zeros((h, w), dtype=np.float32)
    if len(points) == 0: return density
    tree = KDTree(points)
    dists, _ = tree.query(points, k=k+1)
    for i, pt in enumerate(points):
        x, y = int(pt[0]), int(pt[1])
        sigma = max(1.0, beta * np.mean(dists[i][1:]))
        r = int(3 * sigma)
        x_min, x_max = max(0, x - r), min(w, x + r + 1)
        y_min, y_max = max(0, y - r), min(h, y + r + 1)
        px, py = np.meshgrid(np.arange(x_min, x_max) - x, np.arange(y_min, y_max) - y)
        kernel = np.exp(-(px**2 + py**2) / (2 * sigma**2))
        kernel /= np.sum(kernel)
        density[y_min:y_max, x_min:x_max] += kernel
    return density

# Verification test
pts = np.array([[50, 50], [55, 52], [60, 55], [180, 220], [190, 225]])
dmap = generate_density_map(pts, (300, 300))
print(f"Ground truth point count: {len(pts)}")
print(f"Sum of density map integral: {np.sum(dmap):.4f}")